
# Submission Details
* Your Full Name: Bharathiraja Theethappa Muniraju
* Your Uplevel Email Address: bharathiraja.tm@gmail.com
* Name of the Problem Statement of Submission : Group Emotion Detection

## **Business Problem:**
Picture this: You're organizing a company retreat, a school assembly, or perhaps a community gathering. As the organizer, you're keen to ensure everyone is engaged and enjoying themselves. But how can you gauge the collective mood of the group? That's where this project comes in. The aim is to develop a system capable of analyzing group images to identify the emotions of the individuals within them. By employing deep learning models and algorithms, we aim to identify whether the group is demonstrating happiness, enthusiasm, apprehension, or any other emotional state.

Such a system holds immense potential for various applications. For businesses, it could provide valuable insights into employee satisfaction during team-building events or meetings. In educational settings, it could help educators assess student engagement during lectures or group activities. Even event organizers could leverage this tool to ensure attendees are having a positive experience. Ultimately, by decoding group emotions through visual data, we strive to enhance social dynamics and foster environments favorable to collective well-being and productivity. ![img](https://media.springernature.com/lw685/springer-static/image/art%3A10.1186%2Fs40537-020-00326-5/MediaObjects/40537_2020_326_Fig6_HTML.jpg)

## **Existing Solution and Limitations:**

Understanding group emotions from group images is challenging because it involves identifying how each person in the group feels, which can be different for everyone. Traditional methods often treat the whole group as feeling the same way, missing the individual emotions. Also, analyzing group pictures accurately means needing to pick out emotions from both the whole group and each person's face, which can be hard because of different lighting and facial expressions. Plus, emotions can be subtle and vary based on the situation, making it tricky to get it right. Finally, doing this quickly and accurately is important for tasks like managing events or customer interactions.

## **Proposed Solution:**
This project aims to redefine the way we perceive and interpret group emotions by implementing an approach that takes into account the individual emotions within a group. Instead of solely analyzing group images as a whole, our solution involves employing deep learning models to extract emotions of individuals given group images. We strive to create a comprehensive understanding of group emotions that captures the rich diversity of feelings present among group members.


## **Technical Objectives**
### - Face Detection and Extraction:
- Given an image of a group of people, extract and isolate faces of individual people in the image.
- Use pretrained models such as YOLOv8, YOLOv8 Face, Single-Shot Multibox Detector (SSD), or non-ML algorithmic techniques such as HaarCascade to extract these individual faces.

### - Emotion Classification:
- Once the individual faces have been extracted, identify the emotion of each of the faces in the given image.
Use techniques such as majority voting to identify the emotion of the entire group.
- Use Facial Attribute Analysis from DeepFace library to analyze individual emotions. Feel free to explore other Facial Emotion Recognition models.
Labeling, Validation and Evaluation
- To enable performance validation of the pipeline, we have scrapped about 3000 group images from the internet.
- Label all or a subset of images manually for faces and emotions of each of the faces. You can use a tool such as Label Studio for the same. If you choose to label a subset of images, ensure that there is a diversity in the kind of images you include in your subset.
- Use the labeled images to validate the performance of your solution. Benchmark your solution for latency, as well as against statistical metrics such as Intersection of Union (IoU), Accuracy, Precision, Recall etc.

---

[**DeepFace**](https://github.com/serengil/deepface) library comes with support for both - face detection/extraction and Emotion Recognition. However, since it comes with a plethora of models and options (called backends), you need to weigh in the tradeoff between statistical performance and scalability. Explore as many options as you can to ensure that your analysis and solution are comprehensive.
Also, consider exploring [Super Resolution](https://paperswithcode.com/task/image-super-resolution) techniques for improving image quality.

# Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/gdrive")
os.chdir('/content/gdrive/MyDrive/Colab Notebooks')
base_path = './GroupEmotion/'
dataset_path = './GroupEmotion/assets'

# Import

## Setup Environment

In [ ]:
import subprocess
from IPython.display import display, Javascript

def run_pip_command(command: str):
    """
    Runs a pip command with real-time output.
    """
    process = subprocess.Popen(
        command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()


def restart_runtime():
    display(Javascript('''google.colab.kernel.restart()'''))

In [ ]:
def install_and_clean_requirements(use_stored_env: bool, dl_env_file_path: str):
    packages = [
        "numpy",
        "pandas",
        "matplotlib",
        "seaborn",
        "opencv-contrib-python",
        "tqdm",
        "torch",
        "torchvision",
        "torchaudio",
        "ultralytics",
        "deepface",
        "facenet-pytorch",
        "onnxruntime",
        "huggingface_hub"
    ]

    restart_required = False

    if use_stored_env and os.path.exists(dl_env_file_path):
        output = run_pip_command(f'pip install --force-reinstall -r "{dl_env_file_path}"')
        print(f"✅ Installed from saved deep learning environment: {dl_env_file_path}")
    else:
        pip_cmd = (
            "pip install --upgrade --force-reinstall --no-cache-dir " +
            " ".join(packages)
        )
        output = run_pip_command(pip_cmd)
        run_pip_command(f'pip freeze > "{dl_env_file_path}"')

        allowed_packages = {pkg.lower().replace("_", "-") for pkg in packages}

        cleaned_lines = []
        with open(dl_env_file_path, "r") as f:
            for line in f:
                pkg = line.strip().split("==")[0].lower().replace("_", "-")
                if any(pkg.startswith(allowed) for allowed in allowed_packages):
                    cleaned_lines.append(line)

        with open(dl_env_file_path, "w") as f:
            f.writelines(cleaned_lines)

        print(f"✅ Cleaned deep learning requirements saved to: {dl_env_file_path}")

    # Check if a restart is needed based on pip output
    print("\n🔁 Runtime must restart to apply changes.")
    display(Javascript('google.colab.kernel.restart()'))

In [ ]:
install_and_clean_requirements(use_stored_env=True, dl_env_file_path=f"{base_path}/dl_face_env.txt")

In [ ]:
# 🧹 Clean install: latest stable versions of everything
!wget -q https://huggingface.co/deepghs/yolo-face/resolve/main/yolov8n-face/model.pt -O yolov8n-face.pt


In [ ]:
import os, json, time, uuid, random, datetime, pathlib, hashlib, glob, collections, math
import numpy as np
import pandas as pd

import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle
import seaborn as sns

import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from collections import Counter, defaultdict

from deepface import DeepFace, __version__ as deepface_version
from ultralytics import YOLO, __version__ as ultralytics_version
from huggingface_hub import hf_hub_download
import onnxruntime as ort
from facenet_pytorch import InceptionResnetV1

print(f"Ultralytics version: {ultralytics_version}")
print(f"DeepFace version: {deepface_version}")


# Read and Unzip Data

In [ ]:
with os.scandir(base_path) as entries:
    for entry in entries:
        print(entry.name)

In [ ]:
zip_file = os.path.join(dataset_path, "images.zip")
extract_dir = os.path.join(dataset_path, "Unzippeddata")
zip_file, extract_dir

In [ ]:
import zipfile
def extract(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ Extracted to {extract_to}")
    else:
        print(f"⏩ Already extracted at {extract_to}")
extract(zip_file, extract_dir)

# Explore images

In [ ]:
img_extract_path = os.path.join(extract_dir, r"Scraped-Dataset for GroupEmotion")
img_extract_path

In [ ]:
image_paths = glob.glob(os.path.join(img_extract_path, '**/*.jpg'), recursive=True)
print(f"✅ Total images found: {len(image_paths)}")

## Check for duplicates

In [ ]:
def sha1_of_file(path, block=1 << 16):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while chunk := f.read(block):
            h.update(chunk)
    return h.hexdigest()

records = []
for p in tqdm(image_paths):
    try:
        img = cv2.imread(p)              # cv2 is fast & returns None on failure
        if img is None:                  # bad/corrupt file
            raise ValueError("cv2.imread returned None")
        h, w, _ = img.shape
        records.append(
            dict(
                path=p,
                width=w,
                height=h,
                aspect=w / h,
                size_kb=os.path.getsize(p) // 1024,
                sha1=sha1_of_file(p),
            )
        )
    except Exception as e:
        print(f"⚠️ could not read {p}: {e}")

df = pd.DataFrame(records)
print("✅ metadata rows:", len(df))
df.head()

In [ ]:
dupes = df[df.duplicated("sha1", keep=False)]
print(f"Exact duplicates: {len(dupes)} out of {len(df)} images")

In [ ]:
df = df.drop_duplicates("sha1")
image_paths = df["path"].tolist()

# Image Size distribution

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3))

axs[0].hist(df["width"], bins=30)
axs[0].set_title("Width px")

axs[1].hist(df["height"], bins=30)
axs[1].set_title("Height px")

axs[2].hist(df["aspect"], bins=30)
axs[2].set_title("Aspect ratio")

plt.tight_layout()

# Pick few Sample images

In [ ]:
NUM_SAMPLES = min(50, len(image_paths))
sample_paths = random.sample(image_paths, min(NUM_SAMPLES, len(image_paths)))


num_images = len(sample_paths)
cols = min(NUM_SAMPLES, 5)  # Max 5 images per row
rows = math.ceil(NUM_SAMPLES / cols)

# Set up figure size based on layout
plt.figure(figsize=(3 * cols, 3 * rows))

for i, img_path in enumerate(sample_paths):
    img = mpimg.imread(img_path)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Sample {i+1}")

plt.tight_layout()
plt.show()

# Yolo Face Detections Histo


In [ ]:
yolo_detector = YOLO('yolov8n-face.pt')

In [ ]:
face_counts = []
for p in tqdm(image_paths):
    res = yolo_detector(p, imgsz=320, conf=0.3, verbose=False)  # Use detector() directly
    face_counts.append(len(res[0].boxes))  # Access boxes from the result
df["faces"] = face_counts
df["faces"].hist(bins=range(0, df["faces"].max() + 1))
plt.title("Faces per image")

# Sample Yolo Face Detections

In [ ]:
IMG_SIZE    = 640
CONF_THRES  = 0.3

In [ ]:
for img_path in sample_paths:
    # run the face detector (returns a list-like of results; we take the first)
    result = yolo_detector(img_path, imgsz=640, conf=0.3, verbose=False)[0]

    # load the original image (BGR → RGB for Matplotlib)
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    # draw
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    ax.axis("off")

    # result.boxes.xyxy is [N, 4] → x1, y1, x2, y2 in absolute pixels
    for (x1, y1, x2, y2) in result.boxes.xyxy.cpu().numpy():
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2,            # thicker line for visibility
            )
        )

    ax.set_title(f"{img_path.split('/')[-1]}  •  {len(result.boxes)} face(s)")
    plt.show()

# Sample Emotion Detection

In [ ]:
def classify_emotions(crops_bgr):
    results = []
    for crop in crops_bgr:
        if crop.size == 0:
            results.append(("unknown", 0.0))
            continue

        pred = DeepFace.analyze(
            crop,                        # first positional arg = image
            actions           = ["emotion"],
            enforce_detection = False,
            align             = False
        )[0]                             # ⇦ grab the first (and only) dict

        label = pred["dominant_emotion"]
        conf  = pred["emotion"][label]
        results.append((label, conf))
    return results

In [ ]:
group_counter = Counter()

for img_path in sample_paths:
    # 1) Read image
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f"⚠️ Skipped unreadable: {img_path}")
        continue

    # 2) Run YOLOv8 face detector
    result = yolo_detector(img_path, conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)[0]

    # 3) Extract crops
    faces = []
    crops = []
    for (x1, y1, x2, y2) in result.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        crop = img_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        faces.append(dict(box=(x1, y1, x2, y2)))
        crops.append(crop)

    # 4) Classify emotions
    emotions = classify_emotions(crops)
    for f, (emo, conf) in zip(faces, emotions):
        f["emotion"], f["conf"] = emo, conf

    # 5) Group-level emotion (majority vote)
    group_label = Counter(f["emotion"] for f in faces).most_common(1)[0][0] if faces else "no_faces"
    group_counter[group_label] += 1

    # 6) Draw and show
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_rgb)
    ax.axis("off")
    for f in faces:
        x1, y1, x2, y2 = f["box"]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, fill=False))
        ax.text(x1, y1 - 4, f["emotion"][:3], color="white",
                bbox=dict(facecolor="black", alpha=0.6, lw=0, pad=1))
    ax.set_title(f"Group emotion ➜ {group_label}", fontsize=14)
    plt.show()

# ----------------------------
# Final Summary
# ----------------------------
summary_df = (
    pd.Series(group_counter)
    .rename("images")
    .to_frame()
    .sort_values("images", ascending=False)
)
display(summary_df)

summary_df.plot(kind="bar", legend=False, figsize=(4, 2))
plt.title("Group-emotion freq. (YOLO sample)")
plt.ylabel("# images")
plt.tight_layout()

# Sample retinaface detection and emotion classification

In [ ]:
DETECTOR_BACKEND = "retinaface"

def detect_faces(p):
    dets = DeepFace.extract_faces(
        p, detector_backend=DETECTOR_BACKEND,
        enforce_detection=False, align=False)
    fac = []
    for d in dets:
        x,y,w,h = (d["facial_area"][k] for k in ("x","y","w","h"))
        fac.append(dict(box=[x, y, x+w, y+h]))
    return fac

group_hist   = Counter()

for p in sample_paths:
    img_bgr = cv2.imread(str(p))
    faces   = detect_faces(str(p))

    crops = [img_bgr[y1:y2, x1:x2] for x1,y1,x2,y2 in (f["box"] for f in faces)]
    for f, (lab, conf) in zip(faces, classify_emotions(crops)):
        f["emotion"], f["conf"] = lab, conf

    group = Counter(f["emotion"] for f in faces).most_common(1)[0][0] if faces else "no_faces"
    group_hist[group] += 1

    # visualise
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(img_rgb); ax.axis("off")
    for f in faces:
        x1,y1,x2,y2 = f["box"]
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, fill=False))
        ax.text(x1, y1-4, f["emotion"][:3], color="white",
                bbox=dict(facecolor="black", alpha=.6, lw=0, pad=1))
    ax.set_title(f"{pathlib.Path(p).name} • group: {group}")
    plt.show()

print("\nGroup-emotion frequency in sample:", dict(group_hist))

# Combining & Comparing Models

In [ ]:
# -------------------------------------------------------------
# Config
# -------------------------------------------------------------
CONF_THRES     = 0.3
IMG_SIZE       = 640
MAX_DIM        = 5000
MIN_DIM        = 10
MIN_FACE_SIZE  = 0

# -------------------------------------------------------------
# Load models
# -------------------------------------------------------------
yolo_ext = yolo_detector

mediapipe_onnx_path = hf_hub_download(repo_id="qualcomm/MediaPipe-Face-Detection", filename="MediaPipeFaceDetector.onnx")
mp_session = ort.InferenceSession(mediapipe_onnx_path)

attr_session = None  # Facial Attribute Detection ONNX file currently unavailable

facenet = InceptionResnetV1(pretrained='vggface2').eval()
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# -------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------
def resize_if_needed(img):
    h, w = img.shape[:2]
    if max(h, w) > MAX_DIM or min(h, w) < MIN_DIM:
        return None
    if max(h, w) > 800:
        scale = 800 / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
    return img

def classify(crops):
    results = []
    for crop in crops:
        if crop.size == 0:
            results.append(("unknown", 0.0))
            continue
        res = DeepFace.analyze(crop, actions=["emotion"], enforce_detection=False, align=False, silent=True)[0]
        label = res["dominant_emotion"]
        conf  = res["emotion"][label]
        results.append((label, conf))
    return results

def detect_yolo(img, model):
    start = time.time()
    res = model(img, conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)[0]
    boxes = [list(map(int, xy)) for xy in res.boxes.xyxy.cpu().numpy()
             if (xy[2]-xy[0]) >= MIN_FACE_SIZE and (xy[3]-xy[1]) >= MIN_FACE_SIZE]
    crops = [img[y1:y2, x1:x2] for x1, y1, x2, y2 in boxes]
    emotions = classify(crops)
    return [dict(box=b, emotion=e, conf=c) for b, (e, c) in zip(boxes, emotions)], time.time() - start

def detect_mediapipe(img):
    start = time.time()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (256, 256)).astype(np.float32)
    img_input = np.transpose(img_resized, (2, 0, 1))  # CHW
    img_input = np.expand_dims(img_input, axis=0)

    try:
        output = mp_session.run(None, {mp_session.get_inputs()[0].name: img_input})
        box_coords = output[0][0]
        box_scores = output[1][0]
    except Exception as e:
        print(f"MediaPipe inference failed: {e}")
        return [], time.time() - start

    results = []

    for idx, (box, score) in enumerate(zip(box_coords, box_scores)):
        if score < CONF_THRES or len(box) < 4:
            continue

        try:
            x_center, y_center, w, h = box[:4]
            x1 = int(x_center - w / 2)
            y1 = int(y_center - h / 2)
            x2 = int(x_center + w / 2)
            y2 = int(y_center + h / 2)
        except Exception as e:
            print(f"⚠️ Failed to convert box: {box} → {e}")
            continue

        if (x2 - x1) < MIN_FACE_SIZE or (y2 - y1) < MIN_FACE_SIZE or x1 < 0 or y1 < 0 or x2 <= x1 or y2 <= y1:
            continue

        crop = img[y1:y2, x1:x2]
        emotion, conf = classify([crop])[0]
        results.append(dict(box=[x1, y1, x2, y2], emotion=emotion, conf=conf))

    return results, time.time() - start

def detect_attr(img):
    start = time.time()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (256, 256)).astype(np.float32)
    img_input = np.transpose(img_resized, (2, 0, 1))
    img_input = np.expand_dims(img_input, axis=0)
    try:
        output = attr_session.run(None, {attr_session.get_inputs()[0].name: img_input})[0]
    except Exception as e:
        print(f"Attr detection failed: {e}")
        return [], time.time() - start
    return [], time.time() - start

def detect_facenet(img):
    start = time.time()
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    img_tensor = transform(img_pil).unsqueeze(0)
    with torch.no_grad():
        embedding = facenet(img_tensor)
    return [], time.time() - start

def detect_deepface(img, backend):
    start = time.time()
    try:
        dets = DeepFace.extract_faces(img, detector_backend=backend, enforce_detection=False, align=False)
    except Exception as e:
        print(f"⚠️ {backend} failed: {e}")
        return [], 0.0
    boxes = []
    for d in dets:
        x, y, w, h = (d["facial_area"][k] for k in ("x", "y", "w", "h"))
        if w < MIN_FACE_SIZE or h < MIN_FACE_SIZE:
            continue
        boxes.append([x, y, x + w, y + h])
    crops = [img[y1:y2, x1:x2] for x1, y1, x2, y2 in boxes]
    emotions = classify(crops)
    return [dict(box=b, emotion=e, conf=c) for b, (e, c) in zip(boxes, emotions)], time.time() - start

def group_emotion(faces):
    return Counter(f["emotion"] for f in faces).most_common(1)[0][0] if faces else "no_faces"

def draw_faces(ax, img_rgb, faces, title):
    ax.imshow(img_rgb); ax.axis("off")
    for f in faces:
        x1, y1, x2, y2 = f["box"]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2, fill=False))
        ax.text(x1, y1 - 4, f["emotion"][:3], color="white",
                bbox=dict(facecolor="black", alpha=0.6, lw=0, pad=1))
    ax.set_title(title, fontsize=11)

# -------------------------------------------------------------
# Run benchmark
# -------------------------------------------------------------
group_hist = {
    "YOLOv8-ext": Counter(),
    "RetinaFace": Counter(),
    "YUNet": Counter(),
    "SSD": Counter(),
    "MediaPipe": Counter(),
    "AttrDetect": Counter(),
    "FaceNet": Counter()
}

latency_hist = {
    "YOLOv8-ext": [],
    "RetinaFace": [],
    "YUNet": [],
    "SSD": [],
    "MediaPipe": [],
    "AttrDetect": [],
    "FaceNet": []
}

for img_path in sample_paths:
    print(f"\n📷 Processing: {img_path}")
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print("⚠️ Image failed to load."); continue

    img_bgr = resize_if_needed(img_bgr)
    if img_bgr is None:
        print("⚠️ Skipped due to size filter."); continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    faces_yolo, t_yolo = detect_yolo(img_bgr, yolo_ext)
    g_yolo = group_emotion(faces_yolo)
    group_hist["YOLOv8-ext"][g_yolo] += 1
    latency_hist["YOLOv8-ext"].append(t_yolo)

    faces_ret, t_ret = detect_deepface(img_bgr, backend="retinaface")
    g_ret = group_emotion(faces_ret)
    group_hist["RetinaFace"][g_ret] += 1
    latency_hist["RetinaFace"].append(t_ret)

    faces_yun, t_yun = detect_deepface(img_bgr, backend="yunet")
    g_yun = group_emotion(faces_yun)
    group_hist["YUNet"][g_yun] += 1
    latency_hist["YUNet"].append(t_yun)

    faces_ssd, t_ssd = detect_deepface(img_bgr, backend="ssd")
    g_ssd = group_emotion(faces_ssd)
    group_hist["SSD"][g_ssd] += 1
    latency_hist["SSD"].append(t_ssd)

    faces_mp, t_mp = detect_mediapipe(img_bgr)
    g_mp = group_emotion(faces_mp)
    group_hist["MediaPipe"][g_mp] += 1
    latency_hist["MediaPipe"].append(t_mp)

    faces_attr, t_attr = detect_attr(img_bgr)
    g_attr = group_emotion(faces_attr)
    group_hist["AttrDetect"][g_attr] += 1
    latency_hist["AttrDetect"].append(t_attr)

    faces_fn, t_fn = detect_facenet(img_bgr)
    g_fn = group_emotion(faces_fn)
    group_hist["FaceNet"][g_fn] += 1
    latency_hist["FaceNet"].append(t_fn)

    fig, axs = plt.subplots(1, 7, figsize=(42, 5))
    draw_faces(axs[0], img_rgb, faces_yolo, f"YOLOv8-ext ➜ {g_yolo} ({t_yolo:.2f}s)")
    draw_faces(axs[1], img_rgb, faces_ret,  f"RetinaFace ➜ {g_ret} ({t_ret:.2f}s)")
    draw_faces(axs[2], img_rgb, faces_yun,  f"YUNet ➜ {g_yun} ({t_yun:.2f}s)")
    draw_faces(axs[3], img_rgb, faces_ssd,  f"SSD ➜ {g_ssd} ({t_ssd:.2f}s)")
    draw_faces(axs[4], img_rgb, faces_mp,   f"MediaPipe ➜ {g_mp} ({t_mp:.2f}s)")
    draw_faces(axs[5], img_rgb, faces_attr, f"AttrDetect ➜ {g_attr} ({t_attr:.2f}s)")
    draw_faces(axs[6], img_rgb, faces_fn,   f"FaceNet ➜ {g_fn} ({t_fn:.2f}s)")
    plt.suptitle(pathlib.Path(img_path).name, fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# -------------------------------------------------------------
# Summary
# -------------------------------------------------------------
print("\n📊 Group-level emotion frequency across samples:")
for method, counter in group_hist.items():
    print(f"{method.ljust(20)}: {dict(counter)} (Samples: {sum(counter.values())})")

print("\n⏱️ Average Latency per Method (seconds):")
for method, times in latency_hist.items():
    avg_time = np.mean(times) if times else 0.0
    print(f"{method.ljust(20)}: {avg_time:.3f}s")


## Key Takeaways

Based on the evaluation across 49 samples using **CPU**, YOLOv8-ext demonstrates a strong balance between detection accuracy and runtime efficiency:

### Emotion Detection Coverage
YOLOv8-ext successfully detected faces and classified emotions in **100% of the images**, showing a strong emotional spread with 16 sad, 15 neutral, 10 happy, 7 fear, and 1 angry detections.

RetinaFace also covered all samples and exhibited slightly more emotional balance (e.g., higher fear and angry), but its **CPU latency is 17× higher** than YOLOv8-ext, making it impractical without GPU acceleration.

YUNet and SSD both produced complete outputs, though their emotional distributions skewed more toward fear and angry, with lower sad counts, indicating a possible drop in sensitivity to some emotions.

### Performance
YOLOv8-ext ran at an average of **0.528s per image**, maintaining solid detection with significantly faster performance than RetinaFace (9.024s) and acceptable trade-offs compared to faster but less accurate models like YUNet (0.280s) and SSD (0.435s).

MediaPipe, AttrDetect, and FaceNet did not detect any faces across the dataset, despite being faster. These models are therefore unsuitable in this context.

### Comparative Strength
While RetinaFace remains a high-quality detector in terms of emotional diversity, its **high latency on CPU makes it less viable** unless GPU acceleration is available.

YOLOv8-ext offers a strong middle ground: accurate, consistent, and acceptably fast for real-time or batch CPU-based processing. Its complete face detection and balanced emotion spread make it a dependable default.

### Conclusion
On CPU, **YOLOv8-ext is the most reliable and efficient face-emotion pipeline**, offering a strong trade-off between latency, accuracy, and emotional granularity.

RetinaFace is promising in accuracy may be practical with advanced GPU support.

YUNet and SSD are suitable lightweight options if faster inference is prioritized, though with a small trade in emotional detail.

# Evaluation

## PreAnotation Labeling

In [ ]:

# 🔧 CONFIG
SERVER_URL        = "http://192.168.0.63:8888"  # Change to your server path
SCALE_DIMS        = 800
MIN_FACE_SIZE     = 10
SAMPLE_FRAC       = 0.05
BACKEND           = "retinaface"
ALLOWED_EMOTIONS  = {"angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"}

# 📐 IMAGE HELPERS
def resize_if_needed(img, min_dim=10, max_dim=5000):
    h, w = img.shape[:2]
    if max(h, w) > max_dim or min(h, w) < min_dim:
        return None
    if max(h, w) > SCALE_DIMS:
        scale = SCALE_DIMS / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
    return img

def classify(crops):
    out = []
    for crop in crops:
        if crop.size == 0:
            out.append(("unknown", 0.0)); continue
        try:
            res = DeepFace.analyze(crop, actions=["emotion"], enforce_detection=False, align=False, silent=True)[0]
            emo  = res["dominant_emotion"]
            conf = res["emotion"][emo]
            if emo not in ALLOWED_EMOTIONS:
                emo = "unknown"
        except Exception:
            emo, conf = "unknown", 0.0
        out.append((emo, conf))
    return out

def detect_deepface(img, backend):
    try:
        dets = DeepFace.extract_faces(img, detector_backend=backend, enforce_detection=False, align=False)
    except Exception as e:
        print(f"⚠️ {backend} failed: {e}")
        return []
    boxes = []
    for d in dets:
        x, y, w, h = (d["facial_area"][k] for k in ("x", "y", "w", "h"))
        if w < MIN_FACE_SIZE or h < MIN_FACE_SIZE:
            continue
        boxes.append([x, y, x + w, y + h])
    crops = [img[y1:y2, x1:x2] for x1, y1, x2, y2 in boxes]
    emotions = classify(crops)
    return [dict(box=b, emotion=e, conf=c) for b, (e, c) in zip(boxes, emotions)]


In [ ]:
generate_new_output = False

if generate_new_output:
  sampled_image_paths = random.sample(image_paths, max(1, int(SAMPLE_FRAC * len(image_paths))))

  # 🏷️ Generate Label Studio tasks
  # 🏷️ Generate Label Studio tasks
  now_iso = datetime.datetime.utcnow().isoformat() + "Z"
  tasks = []

  for img_path in tqdm(sampled_image_paths, desc="Processing"):
      img_bgr = cv2.imread(img_path)
      if img_bgr is None:
          continue

      img_bgr = resize_if_needed(img_bgr)
      if img_bgr is None:
          continue

      h, w = img_bgr.shape[:2]
      detections = detect_deepface(img_bgr, BACKEND)
      results = []

      for det in detections:
          x1, y1, x2, y2 = det["box"]
          emotion = det["emotion"]
          conf    = det["conf"]
          if emotion not in ALLOWED_EMOTIONS:
              continue

          uid = str(uuid.uuid4())  # Shared ID for both box and emotion
          x_pct = 100 * x1 / w
          y_pct = 100 * y1 / h
          width_pct  = 100 * (x2 - x1) / w
          height_pct = 100 * (y2 - y1) / h

          common_value = {
              "x": x_pct,
              "y": y_pct,
              "width": width_pct,
              "height": height_pct,
              "rotation": 0,
              "rectanglelabels": ["Face"]
          }

          # Rectangle label
          results.append({
              "id": uid,
              "type": "rectanglelabels",
              "from_name": "box",
              "to_name": "img",
              "origin": "manual",
              "image_rotation": 0,
              "original_width": w,
              "original_height": h,
              "score": float(conf),
              "value": common_value
          })

          # Emotion label
          emotion_value = common_value.copy()
          emotion_value["choices"] = [emotion]
          results.append({
              "id": uid,
              "type": "choices",
              "from_name": "emotion",
              "to_name": "img",
              "origin": "manual",
              "image_rotation": 0,
              "original_width": w,
              "original_height": h,
              "value": emotion_value
          })

      if results:
          annotation = {
              "completed_by": 1,
              "created_at": now_iso,
              "updated_at": now_iso,
              "lead_time": 0,
              "was_cancelled": False,
              "ground_truth": False,
              "result": results
          }
          tasks.append({
              "data": {
                  "image": f"{SERVER_URL}/{os.path.basename(img_path)}"
              },
              "annotations": [annotation]
          })

  output_path = os.path.join(dataset_path, "preannotate_faces_with_emotions.json")

  try:
      with open(output_path, "w") as f:
          json.dump(tasks, f, indent=2)
      print(f"✅ Done! {len(tasks)} tasks saved to {output_path}")
  except Exception as e:
      print(f"❌ Error writing JSON: {e}")

## Import updated labels

In [ ]:
ground_truth_path = f"{dataset_path}/annotated_faces_with_emotions.json"

# Load data
with open(ground_truth_path) as f:
    ground_truth = json.load(f)

# Helper to convert % bbox to absolute coordinates
def to_absolute_bbox(value, width, height):
    x = int(value['x'] / 100 * width)
    y = int(value['y'] / 100 * height)
    w = int(value['width'] / 100 * width)
    h = int(value['height'] / 100 * height)
    return [x, y, w, h]


## Metrics

#### Ground Truths

In [ ]:
def extract_faces(data):
    face_map = defaultdict(list)
    for task in data:
        image_url = task['data']['image']
        image_name = os.path.basename(image_url)
        ann = task['annotations'][0]['result']
        width = ann[0]['original_width']
        height = ann[0]['original_height']
        temp = {}
        for r in ann:
            if r['type'] == 'rectanglelabels':
                temp[r['id']] = {'bbox': to_absolute_bbox(r['value'], width, height)}
            elif r['type'] == 'choices' and r['id'] in temp:
                temp[r['id']]['emotion'] = r['value']['choices'][0]
        face_map[image_name] = [v for v in temp.values() if 'emotion' in v]
    return face_map

gt_faces = extract_faces(ground_truth)

#### Yolo Predictions (basis for production pipeline)

In [ ]:
def get_predictions(image_bgr, model, conf_thres=0.3):
    results = model(image_bgr, conf=conf_thres, imgsz=640, verbose=False)[0]
    boxes = [list(map(int, xy)) for xy in results.boxes.xyxy.cpu().numpy()]
    faces = []
    for x1, y1, x2, y2 in boxes:
        face = image_bgr[y1:y2, x1:x2]
        if face.size == 0:
            continue
        try:
            res = DeepFace.analyze(face, actions=["emotion"], enforce_detection=False, align=False, silent=True)[0]
            faces.append({"bbox": [x1, y1, x2 - x1, y2 - y1], "emotion": res["dominant_emotion"]})
        except Exception:
            continue
    return faces

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]
    return interArea / float(boxAArea + boxBArea - interArea)

y_true = []
y_pred = []
y_true_grouped = []
y_pred_grouped = []


for image_name, gt_list in gt_faces.items():
    image_path = os.path.join(img_extract_path, image_name)
    if not os.path.exists(image_path):
        continue
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        continue

    pred_list = get_predictions(img_bgr, yolo_detector)
    matched = set()
    for gt in gt_list:
        best_iou = 0
        best_pred = None
        for idx, pred in enumerate(pred_list):
            if idx in matched:
                continue
            iou_score = iou(gt['bbox'], pred['bbox'])
            if iou_score > best_iou:
                best_iou = iou_score
                best_pred = (idx, pred)
        if best_iou >= 0.5 and best_pred:
            matched.add(best_pred[0])
            y_true.append(gt['emotion'])
            y_pred.append(best_pred[1]['emotion'])

            # Save for group evaluation
            y_true_grouped.append((image_name, gt['emotion']))
            y_pred_grouped.append((image_name, best_pred[1]['emotion']))

In [ ]:
import pickle

results_file = f"{dataset_path}/y_true_y_pred.pkl"

with open(results_file, "wb") as f:
    pickle.dump({
        "face_level": (y_true, y_pred),
        "group_level": (y_true_grouped, y_pred_grouped)
    }, f)

print(f"✅ Results saved to: {results_file}")


#### Sklearn Environment for Model Evaluation

In [ ]:
import os
import subprocess

def install_fixed_sklearn_env(use_stored_env: bool, sklearn_env_file_path: str):
    allowed_packages = {"numpy", "scikit-learn"}

    output = ""

    if use_stored_env and os.path.exists(sklearn_env_file_path):
        output = run_pip_command(f'pip install --force-reinstall -r "{sklearn_env_file_path}"')
        print(f"✅ Installed from saved sklearn environment: {sklearn_env_file_path}")
    else:
        # Step 1: Install fixed versions
        output = run_pip_command("pip install --force-reinstall numpy==1.26.4 scikit-learn==1.4.2")

        # Step 2: Save all installed packages
        run_pip_command(f'pip freeze > "{sklearn_env_file_path}"')

        # Step 3: Filter out only allowed packages
        cleaned_lines = []
        with open(sklearn_env_file_path, "r") as f:
            for line in f:
                pkg = line.strip().split("==")[0].lower().replace("_", "-")
                if any(pkg.startswith(apkg.lower().replace("_", "-")) for apkg in allowed_packages):
                    cleaned_lines.append(line)

        with open(sklearn_env_file_path, "w") as f:
            f.writelines(cleaned_lines)

        print(f"✅ Cleaned sklearn requirements saved to: {sklearn_env_file_path}")

    # Step 4: Check for runtime restart requirement
    print("\n🔁 Runtime must restart to apply changes.")
    display(Javascript('google.colab.kernel.restart()'))


In [ ]:
install_fixed_sklearn_env(use_stored_env=True, sklearn_env_file_path=f"{base_path}/sklearn_env.txt")

In [ ]:
import os
from google.colab import drive
drive.mount("/content/gdrive")
os.chdir('/content/gdrive/MyDrive/Colab Notebooks')
base_path = './GroupEmotion/'
dataset_path = './GroupEmotion/assets'

In [ ]:
import pickle
results_file = f"{dataset_path}/y_true_y_pred.pkl"

with open(results_file, "rb") as f:
    data = pickle.load(f)

y_true, y_pred = data["face_level"]
y_true_grouped, y_pred_grouped = data["group_level"]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
print("Recall:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
print("F1 Score:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

labels = sorted(list(set(y_true + y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Emotion Prediction Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
from collections import defaultdict, Counter

# Group emotions per image
true_dict = defaultdict(list)
pred_dict = defaultdict(list)

for img, label in y_true_grouped:
    true_dict[img].append(label)
for img, label in y_pred_grouped:
    pred_dict[img].append(label)

group_true = []
group_pred = []

for img in sorted(set(true_dict) & set(pred_dict)):
    gt_major = Counter(true_dict[img]).most_common(1)[0][0]
    pd_major = Counter(pred_dict[img]).most_common(1)[0][0]
    group_true.append(gt_major)
    group_pred.append(pd_major)

# Metrics
print("\nGroup-Level Evaluation")
print("Accuracy:", accuracy_score(group_true, group_pred))
print("Precision:", precision_score(group_true, group_pred, average='weighted', zero_division=0))
print("Recall:", recall_score(group_true, group_pred, average='weighted', zero_division=0))
print("F1 Score:", f1_score(group_true, group_pred, average='weighted', zero_division=0))

# Confusion Matrix
group_labels = sorted(list(set(group_true + group_pred)))
cm = confusion_matrix(group_true, group_pred, labels=group_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=group_labels, yticklabels=group_labels)
plt.title('Group-Level Emotion Prediction Confusion Matrix')
plt.ylabel('True Group Emotion')
plt.xlabel('Predicted Group Emotion')
plt.tight_layout()
plt.show()


# Conclusion

Group Emotion Summary Report
Pipeline: DeepFace (Emotion Classification) + YOLOv8n-face (Face Detection)

## Strengths:
- Fast face detection with low latency, ideal for real-time or large-scale processing.

- Good emotion classification under optimal conditions (frontal, adult, high-contrast faces).

## Limitations:
- Misses small background faces: YOLOv8n-face may struggle with very small or distant faces, especially in cluttered group scenes.

- Top-view angle issues: Emotion predictions are less reliable when faces are viewed from above.

- Struggles with children’s faces: Facial expressions of kids are frequently misclassified, likely due to dataset limitations.

- Happy vs. fear confusion: Overlapping facial cues (e.g., wide eyes, open mouth) cause frequent mislabeling.

- Sad often labeled neutral: Low-intensity sad expressions — especially with closed mouths — are often mistaken as neutral.

- Bias against dark-toned faces: Misclassification is more common, particularly for closed-mouth expressions (neutral/sad), due to reduced contrast and possible training bias.

- False positives from posters/photos: Faces in static images (e.g., posters, screens) are not filtered out.

- Low-contrast face issues: Poor lighting leads to misclassifications as fear or sadness due to loss of visual detail.

